# Product Recommendation Agent System (DuckDuckGo Search)

### What are we building?
A system with **4 AI agents** that work together to recommend the best product for you.

### Why 4 agents instead of 1?
Each agent is an **expert** at one thing:
- Agent 0 = **Scout** → Finds products on the internet
- Agent 1 = **Engineer** → Researches technical specs
- Agent 2 = **Critic** → Reads and analyzes reviews
- Agent 3 = **Advisor** → Scores everything, gives final recommendation

### Flow:
```
Your Query → Agent 0 (find products) → Agent 1 (get specs) → Agent 2 (get reviews) → Agent 3 (rank & recommend)
```

### Tech Stack
- **Pydantic** — Forces LLM to return structured data
- **LangChain** — Connects prompts to LLMs
- **DuckDuckGo** — Free web search (**no API key needed!**)
- **OpenAI GPT** — The brain (LLM)

**Constraint**: No orchestration frameworks — pure Python.

---
# PART 1: Setup & Configuration

### Difference from Tavily version:
- Tavily needs `TAVILY_API_KEY` → DuckDuckGo needs **nothing** for search
- Only `OPENAI_API_KEY` is required here

In [ ]:
# Uncomment and run ONCE
# !pip install langchain langchain-openai langchain-community pydantic python-dotenv duckduckgo-search

In [ ]:
import os
from datetime import date
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults  # FREE search — no API key!

In [ ]:
load_dotenv(override=True)

# Only OPENAI_API_KEY needed — no search API key!
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# DuckDuckGoSearchResults = LangChain's built-in DuckDuckGo wrapper
# num_results = how many results to return per search
ddg_search = DuckDuckGoSearchResults(num_results=10)

print("Setup complete! (DuckDuckGo — no search API key needed)")

---
# PART 2: Pydantic Models (Structured Output)

### Same as Tavily version — schemas don't depend on search tool
Pydantic defines **what format** the LLM returns. The search tool only affects **what data** gets fed in.

In [ ]:
# ======== AGENT 0: Discovery ========
class DiscoveredProduct(BaseModel):
    brand: str = Field(description="Product manufacturer (e.g., ASUS, Lenovo)")
    model: str = Field(description="Product model name (e.g., ROG Strix G16)")
    specs_hint: str = Field(description="Brief specs from search (e.g., RTX 4060, 16GB)")
    source: str = Field(description="URL where this product was found")
    reason: str = Field(description="Why this product matches the user query")

class DiscoveryOutput(BaseModel):
    category: str = Field(description="Detected category (e.g., Laptops)")
    budget: str = Field(description="Detected budget (e.g., Under 1,50,000)")
    use_case: str = Field(description="Detected use case (e.g., AI Development)")
    products: List[DiscoveredProduct] = Field(description="List of products (up to 5)")

# ======== AGENT 1: Specifications ========
class TechnicalSpecs(BaseModel):
    product_name: str = Field(description="Full product name (Brand + Model)")
    cpu: str = Field(description="Processor model and speed")
    gpu: str = Field(description="Graphics card model and VRAM")
    ram: str = Field(description="RAM capacity and type")
    storage: str = Field(description="Storage capacity and type")
    display: str = Field(description="Display size, resolution, panel type")
    battery: str = Field(description="Battery capacity and estimated life")
    weight: str = Field(description="Device weight in kg")
    price: str = Field(description="Approximate price in INR")
    summary: str = Field(description="One-line AI/ML focused summary")

class SpecificationOutput(BaseModel):
    specs: List[TechnicalSpecs] = Field(description="Specs for all products")

# ======== AGENT 2: Reviews ========
class ProductReview(BaseModel):
    product_name: str = Field(description="Full product name")
    overall_sentiment: str = Field(description="Positive, Mixed, or Negative")
    average_rating: str = Field(description="Estimated rating out of 5")
    pros: List[str] = Field(description="Top 3-5 praised features")
    cons: List[str] = Field(description="Top 3-5 common complaints")
    common_issues: List[str] = Field(description="Recurring problems")
    expert_opinion: str = Field(description="Expert reviewer summary")

class ReviewOutput(BaseModel):
    reviews: List[ProductReview] = Field(description="Review analysis for all products")

# ======== AGENT 3: Recommendation ========
class ProductScore(BaseModel):
    product_name: str = Field(description="Full product name")
    performance_score: float = Field(description="Hardware capability (0-10)")
    value_score: float = Field(description="Value for money (0-10)")
    review_score: float = Field(description="User satisfaction (0-10)")
    ai_readiness_score: float = Field(description="AI/ML suitability (0-10)")
    overall_score: float = Field(description="Weighted overall (0-10)")
    rank: int = Field(description="Final rank (1 = best)")

class RecommendationOutput(BaseModel):
    top_pick: str = Field(description="#1 recommended product")
    top_pick_reason: str = Field(description="Why it's the top pick")
    scores: List[ProductScore] = Field(description="Scores for all products")
    trade_offs: str = Field(description="Trade-offs between top products")
    final_verdict: str = Field(description="Final advice (2-3 sentences)")
    confidence: str = Field(description="High, Medium, or Low")

print("All 4 agent schemas ready!")

### Experiment: Test Pydantic

In [ ]:
test = DiscoveredProduct(brand="ASUS", model="ROG Strix G16", specs_hint="RTX 4060",
                         source="https://example.com", reason="Great for AI")
print("Object:", test)
print("Dict:", test.model_dump())
print("Brand:", test.brand)

---
# PART 3: Search Tools (DuckDuckGo)

### How DuckDuckGo search works here:
```python
ddg_search = DuckDuckGoSearchResults(num_results=10)
results = ddg_search.run("best laptops 2026")   # Returns string of results
```

### Difference from Tavily:
- **Tavily**: Returns structured JSON with title, URL, content, score → need to format
- **DuckDuckGo**: Returns a pre-formatted string → simpler but less structured

In [ ]:
def search_products(query: str) -> str:
    """Search using DuckDuckGo. Returns results as string."""
    try:
        return ddg_search.run(query)
    except Exception as e:
        print(f"Search error: {e}")
        return ""

def search_specs(product_name: str) -> str:
    """Search for technical specs."""
    return search_products(f"{product_name} full technical specifications price India {date.today().year}")

def search_reviews(product_name: str) -> str:
    """Search for reviews."""
    return search_products(f"{product_name} review pros cons user experience {date.today().year}")

print("DuckDuckGo search tools ready.")

### Experiment: Test DuckDuckGo search

In [ ]:
# EXPERIMENT: Try a search
test_results = search_products("best laptops for AI development 2026 India")
print(test_results)

---
# PART 4–7: All 4 Agents (Build & Run)

### Same agent logic as Tavily version
The agents don't care which search tool you use. They just need **text data** as input.

**Pattern for each agent:**
1. Build prompt template
2. Create chain = `prompt | llm.with_structured_output(Schema)`
3. Search for data
4. `chain.invoke({...})` → returns Pydantic object

In [ ]:
# ======== BUILD ALL 4 AGENTS ========

# Agent 0: Discovery
discovery_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product discovery expert. Extract up to 5 real product models from search results. Ignore generic articles."),
    ("human", "User Query:\n{user_query}\n\nSearch Results:\n{search_context}")
])
discovery_chain = discovery_prompt | llm.with_structured_output(DiscoveryOutput)

# Agent 1: Specifications
spec_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a hardware spec expert. Extract CPU, GPU, RAM, Storage, Display, Battery, Weight, Price. Write 'Not available' if missing."),
    ("human", "Products:\n{product_list}\n\nResearch Data:\n{research_data}")
])
spec_chain = spec_prompt | llm.with_structured_output(SpecificationOutput)

# Agent 2: Reviews
review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a review analysis expert. Determine sentiment, rating/5, pros, cons, issues, expert opinion for each product."),
    ("human", "Products:\n{product_list}\n\nReview Data:\n{review_data}")
])
review_chain = review_prompt | llm.with_structured_output(ReviewOutput)

# Agent 3: Recommendation
rec_prompt = ChatPromptTemplate.from_messages([
    ("system", "Score each product (0-10): Performance(30%), Value(25%), Reviews(20%), AI Readiness(25%). Rank, give top pick, trade-offs, verdict, confidence."),
    ("human", "User: {user_query}\n\nProducts:\n{discovery_data}\n\nSpecs:\n{spec_data}\n\nReviews:\n{review_data}")
])
recommendation_chain = rec_prompt | llm.with_structured_output(RecommendationOutput)

print("All 4 agents built!")

---
## Run Agent 0 — Discovery

In [ ]:
user_query = "Best laptops for AI development under 150000 INR"
print(f"Query: {user_query}\n[Agent 0] Searching...")

search_context = search_products(f"best {user_query} top models India {date.today().year}")
discovery_result = discovery_chain.invoke({"user_query": user_query, "search_context": search_context})

print(f"Category: {discovery_result.category} | Budget: {discovery_result.budget}")
for i, p in enumerate(discovery_result.products, 1):
    print(f"  {i}. {p.brand} {p.model} — {p.specs_hint}")

## Run Agent 1 — Specifications

In [ ]:
print("[Agent 1] Researching specs...")
product_list_str = "\n".join(f"- {p.brand} {p.model}" for p in discovery_result.products)

all_research = ""
for p in discovery_result.products:
    name = f"{p.brand} {p.model}"
    print(f"  Searching: {name}")
    all_research += f"\n--- {name} ---\n{search_specs(name)}\n"

spec_result = spec_chain.invoke({"product_list": product_list_str, "research_data": all_research})

for s in spec_result.specs:
    print(f"\n{s.product_name}: CPU={s.cpu}, GPU={s.gpu}, RAM={s.ram}, Price={s.price}")

---
# Understanding Sentiment Analysis

Before running Agent 2, let's understand **what sentiment analysis is**.

## What is it?
Determining whether text expresses a **positive, negative, or neutral** opinion.

```
"Amazing laptop! Great GPU performance."  → POSITIVE
"Terrible build quality. Overheats fast."  → NEGATIVE
"It's okay for the price. Nothing special." → MIXED
```

## 3 Approaches:

**1. Rule-Based (Old)**: Count positive/negative words. Fails on sarcasm: *"Not bad"* → thinks negative!

**2. ML Classification**: Train model on labeled data. Needs lots of training data.

**3. LLM-Based (What we use!) ✅**: Ask the LLM directly. Understands context, sarcasm, nuance. No training data needed. Can also extract WHY (pros, cons) — not just positive/negative.

## Types of Sentiment Analysis:
- **Binary**: Positive or Negative
- **Ternary**: Positive, Negative, or Neutral
- **Fine-grained**: 1-5 star rating
- **Aspect-based**: Sentiment per feature ("Battery: Negative, Display: Positive")

**Our project uses**: Ternary sentiment + Fine-grained rating + Aspect-based extraction (pros/cons)

In [ ]:
# EXPERIMENT: Try sentiment analysis yourself!

class SentimentResult(BaseModel):
    text: str = Field(description="The original review text")
    sentiment: str = Field(description="Positive, Negative, or Mixed")
    confidence: float = Field(description="Confidence 0.0 to 1.0")
    key_phrases: List[str] = Field(description="Key phrases indicating sentiment")

sentiment_chain = llm.with_structured_output(SentimentResult)

sample_reviews = [
    "This laptop is incredible! RTX 4060 handles AI workloads perfectly. Battery could be better.",
    "Terrible experience. Screen flickered after 2 weeks. Support was unhelpful.",
    "Decent for the price. Nothing extraordinary but gets the job done for basic ML.",
]

print("SENTIMENT ANALYSIS EXPERIMENT")
print("=" * 50)
for review in sample_reviews:
    r = sentiment_chain.invoke(f"Analyze sentiment:\n{review}")
    print(f"\n📝 {r.text[:50]}...")
    print(f"   Sentiment: {r.sentiment} | Confidence: {r.confidence}")
    print(f"   Key Phrases: {r.key_phrases}")

# TRY: Add sarcastic reviews or Hinglish reviews!

---
## Run Agent 2 — Review Analysis

Now Agent 2 applies sentiment analysis **at scale** across all products.

In [ ]:
print("[Agent 2] Analyzing reviews...")
all_reviews = ""
for p in discovery_result.products:
    name = f"{p.brand} {p.model}"
    print(f"  Searching reviews: {name}")
    all_reviews += f"\n--- {name} ---\n{search_reviews(name)}\n"

review_result = review_chain.invoke({"product_list": product_list_str, "review_data": all_reviews})

for r in review_result.reviews:
    print(f"\n{r.product_name} — {r.overall_sentiment} ({r.average_rating}/5)")
    print(f"  Pros: {', '.join(r.pros[:3])}")
    print(f"  Cons: {', '.join(r.cons[:3])}")

## Run Agent 3 — Final Recommendation

In [ ]:
print("[Agent 3] Generating recommendation...")

disc_str = "\n".join(f"- {p.brand} {p.model}: {p.specs_hint}" for p in discovery_result.products)
spec_str = "\n".join(f"- {s.product_name}: CPU={s.cpu}, GPU={s.gpu}, RAM={s.ram}, Price={s.price}" for s in spec_result.specs)
rev_str = "\n".join(f"- {r.product_name}: {r.overall_sentiment}, {r.average_rating}/5" for r in review_result.reviews)

recommendation_result = recommendation_chain.invoke({
    "user_query": user_query, "discovery_data": disc_str,
    "spec_data": spec_str, "review_data": rev_str
})

print("\n" + "=" * 70 + "\nFINAL RECOMMENDATION\n" + "=" * 70)
print(f"🏆 TOP PICK: {recommendation_result.top_pick}")
print(f"   {recommendation_result.top_pick_reason}")
for s in sorted(recommendation_result.scores, key=lambda x: x.rank):
    print(f"  #{s.rank} {s.product_name} — {s.overall_score}/10")
print(f"⚖️  {recommendation_result.trade_offs}")
print(f"✅ {recommendation_result.final_verdict}")
print(f"🎯 Confidence: {recommendation_result.confidence}")

---
# PART 8: Full Pipeline — One Function

In [ ]:
def run_product_recommendation(query: str) -> RecommendationOutput:
    """Run ALL 4 agents in sequence using DuckDuckGo search."""
    print("=" * 70 + f"\nQuery: {query}\n" + "=" * 70)

    # Agent 0
    print("\n[Agent 0] Discovering...")
    ctx = search_products(f"best {query} top models India {date.today().year}")
    disc = discovery_chain.invoke({"user_query": query, "search_context": ctx})
    for p in disc.products: print(f"  - {p.brand} {p.model}")

    # Agent 1
    print("\n[Agent 1] Specs...")
    pl = "\n".join(f"- {p.brand} {p.model}" for p in disc.products)
    rd = ""
    for p in disc.products:
        n = f"{p.brand} {p.model}"; print(f"  {n}")
        rd += f"\n--- {n} ---\n{search_specs(n)}\n"
    sp = spec_chain.invoke({"product_list": pl, "research_data": rd})

    # Agent 2
    print("\n[Agent 2] Reviews...")
    rv = ""
    for p in disc.products:
        n = f"{p.brand} {p.model}"; print(f"  {n}")
        rv += f"\n--- {n} ---\n{search_reviews(n)}\n"
    rev = review_chain.invoke({"product_list": pl, "review_data": rv})

    # Agent 3
    print("\n[Agent 3] Recommending...")
    ds = "\n".join(f"- {p.brand} {p.model}: {p.specs_hint}" for p in disc.products)
    ss = "\n".join(f"- {s.product_name}: CPU={s.cpu}, GPU={s.gpu}, RAM={s.ram}, Price={s.price}" for s in sp.specs)
    rs = "\n".join(f"- {r.product_name}: {r.overall_sentiment}, {r.average_rating}/5" for r in rev.reviews)
    rec = recommendation_chain.invoke({"user_query": query, "discovery_data": ds, "spec_data": ss, "review_data": rs})

    print("\n" + "=" * 70 + "\nFINAL REPORT\n" + "=" * 70)
    print(f"🏆 {rec.top_pick} — {rec.top_pick_reason}")
    for s in sorted(rec.scores, key=lambda x: x.rank):
        print(f"  #{s.rank} {s.product_name} — {s.overall_score}/10")
    print(f"⚖️  {rec.trade_offs}")
    print(f"✅ {rec.final_verdict}")
    print(f"🎯 Confidence: {rec.confidence}")
    return rec

print("Pipeline ready!")

In [ ]:
# RUN the full pipeline
result = run_product_recommendation("Best laptops for AI development under 150000 INR")

In [ ]:
# EXPERIMENT: Access results
print("Top:", result.top_pick)
print("Confidence:", result.confidence)
print("Dict:", result.model_dump())

### Experiment: Try different queries!
```python
run_product_recommendation("Best headphones under 5000 INR")
run_product_recommendation("Best monitor for coding under 25000 INR")
run_product_recommendation("Best mobile under 20000 INR with good camera")
```